In [1]:
# ============================================================
# FINAL BULLETPROOF SCRIPT: Student-Independent Sign Language
# ============================================================

import os
import time
import pickle
import json
import numpy as np
import pandas as pd

from collections import Counter
from sklearn.preprocessing import StandardScaler, LabelEncoder

import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    LSTM,
    Dense,
    Dropout,
    BatchNormalization,
    Bidirectional,
    Attention,
    GlobalAveragePooling1D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

# ============================================================
# 1. CONFIGURATION & PATHS
# ============================================================

BASE_DIR = r"D:\aaa EAAI Major Revision\dynamic\dynamic_dataset"
DATA_DIR = os.path.join(BASE_DIR, "csv_dataset")

TRAIN_PATH = os.path.join(DATA_DIR, "train_csv")
VAL_PATH = os.path.join(DATA_DIR, "validation_csv")
TEST_PATH = os.path.join(DATA_DIR, "test_csv")

RESULTS_DIR = os.path.join(BASE_DIR, "results_130features_120frames")
os.makedirs(RESULTS_DIR, exist_ok=True)

SEQUENCE_LENGTH = 120
NUM_FEATURES = 130
EXPECTED_CLASSES = 40

BATCH_SIZE = 64
EPOCHS = 200
LEARNING_RATE = 0.0005

np.random.seed(42)
tf.random.set_seed(42)

# ============================================================
# 2. SAFE DATA LOADING PIPELINE
# ============================================================

def load_sequences_safely(base_path):
    if not os.path.exists(base_path):
        raise FileNotFoundError(f"Directory missing: {base_path}")
        
    class_names = sorted([
        d for d in os.listdir(base_path)
        if os.path.isdir(os.path.join(base_path, d))
    ])

    sequences = []
    labels = []

    print(f"\nScanning: {base_path} ({len(class_names)} classes found)")

    for class_name in class_names:
        class_path = os.path.join(base_path, class_name)
        video_names = sorted([
            d for d in os.listdir(class_path)
            if os.path.isdir(os.path.join(class_path, d))
        ])

        for video_name in video_names:
            video_path = os.path.join(class_path, video_name)
            frames = []
            valid = True

            for frame_num in range(1, SEQUENCE_LENGTH + 1):
                frame_file = os.path.join(video_path, f"{frame_num:03d}.npy")
                
                if not os.path.exists(frame_file):
                    valid = False
                    break
                
                try:
                    frame = np.load(frame_file).astype(np.float32)
                except Exception:
                    valid = False
                    break

                if frame.shape != (NUM_FEATURES,):
                    valid = False
                    break

                frames.append(frame)

            if valid and len(frames) == SEQUENCE_LENGTH:
                sequences.append(np.stack(frames))
                labels.append(class_name)

    return np.array(sequences, dtype=np.float32), np.array(labels), class_names

# Load all splits
X_train, y_train, train_classes = load_sequences_safely(TRAIN_PATH)
X_val, y_val, val_classes = load_sequences_safely(VAL_PATH)
X_test, y_test, test_classes = load_sequences_safely(TEST_PATH)

print(f"\nShapes Loaded -> Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# ============================================================
# 3. ENCODING & SCALING
# ============================================================

actions = sorted(list(set(train_classes) | set(val_classes) | set(test_classes)))
if len(actions) != EXPECTED_CLASSES:
    print(f"Warning: Found {len(actions)} classes, expected {EXPECTED_CLASSES}.")

label_encoder = LabelEncoder()
label_encoder.fit(actions)

y_train_enc = label_encoder.transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

# Save encoder artifacts
with open(os.path.join(RESULTS_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(label_encoder, f)

# Flatten for Scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.reshape(-1, NUM_FEATURES)).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val.reshape(-1, NUM_FEATURES)).reshape(X_val.shape)
X_test_scaled = scaler.transform(X_test.reshape(-1, NUM_FEATURES)).reshape(X_test.shape)

with open(os.path.join(RESULTS_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

y_train_cat = tf.keras.utils.to_categorical(y_train_enc, num_classes=len(actions))
y_val_cat = tf.keras.utils.to_categorical(y_val_enc, num_classes=len(actions))
y_test_cat = tf.keras.utils.to_categorical(y_test_enc, num_classes=len(actions))

# ============================================================
# 4. MODEL ARCHITECTURE (CONV1D + BiLSTM + ATTENTION)
# ============================================================

inp = Input(shape=(SEQUENCE_LENGTH, NUM_FEATURES))
x = Conv1D(64, kernel_size=3, activation="relu", padding="same")(inp)
x = BatchNormalization()(x)

x = Bidirectional(LSTM(128, return_sequences=True, dropout=0.2))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

query = Dense(64)(x)
value = Dense(64)(x)
x = Attention()([query, value])

x = GlobalAveragePooling1D()(x)
x = Dense(128, activation="relu", kernel_regularizer=l2(1e-5))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

out = Dense(len(actions), activation="softmax")(x)
model = Model(inputs=inp, outputs=out)

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# ============================================================
# 5. TRAINING EXECUTION
# ============================================================

best_model_path = os.path.join(RESULTS_DIR, "best_model.keras")

callbacks = [
    EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=1),
    ModelCheckpoint(best_model_path, monitor="val_accuracy", save_best_only=True, mode="max", verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10, min_lr=1e-6, verbose=1)
]

history = model.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_val_scaled, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# Save metadata for UI / Deployment
with open(os.path.join(RESULTS_DIR, "classes.json"), "w", encoding="utf-8") as f:
    json.dump(actions, f, indent=4)

print("\nTraining completed successfully! Model and checkpoints saved.")


Scanning: D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\csv_dataset\train_csv (40 classes found)

Scanning: D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\csv_dataset\validation_csv (40 classes found)

Scanning: D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\csv_dataset\test_csv (40 classes found)

Shapes Loaded -> Train: (6179, 120, 130), Val: (1275, 120, 130), Test: (1212, 120, 130)
Epoch 1/200
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 415ms/step - accuracy: 0.1295 - loss: 3.4195
Epoch 1: val_accuracy improved from None to 0.29961, saving model to D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\results_130features_120frames\best_model.keras

Epoch 1: finished saving model to D:\aaa EAAI Major Revision\dynamic\dynamic_dataset\results_130features_120frames\best_model.keras
97/97 ━━━━━━━━━━━━━━━━━━━━ 61s 471ms/step - accuracy: 0.1295 - loss: 3.4195 - val_accuracy: 0.2996 - val_loss: 3.0471 - learning_rate: 5.0000e-04
Epoch 2/200
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 425ms/step - accuracy: 